## 1. 먼저 데이터 처리를 위한 라이브러리를 불러오고, HTML 및 PDF 파일이 저장된 경로를 설정한다.

In [ ]:
import sys
!{sys.executable} -m pip install lxml beautifulsoup4 html5lib pdfplumber

In [ ]:
import re
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import pdfplumber

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

BASE_DIR = Path(".")
HTML_DIR = BASE_DIR / "html"
PDF_DIR = BASE_DIR / "pdf"

CURRENCIES = ["USD", "EUR", "JPY"]

PDF 파일은 HTML보다 구조가 불안정하므로, 먼저 텍스트를 줄 단위로 추출한 뒤 정규표현식을 이용하여 날짜와 환율 값을 찾아낸다.

이때 우리은행 환율표는 일반적으로 한 행에 여러 값이 함께 출력되므로, 그중 매매기준율에 해당하는 값을 선택하여 사용한다.

In [ ]:
print("HTML_DIR exists:", HTML_DIR.exists())
print("PDF_DIR exists:", PDF_DIR.exists())

이 단계에서는 USD.html, EUR.html, JPY.html 파일을 각각 읽어와 날짜 기준으로 병합하여 하나의 HTML 기반 통합 데이터프레임을 생성한다.

이 데이터프레임의 행 인덱스는 날짜이며, 컬럼명은 USD, EUR, JPY이다.

In [ ]:
print("HTML files:")
for p in HTML_DIR.iterdir():
    print(p.name)

print("\nPDF files:")
for p in PDF_DIR.iterdir():
    print(p.name)

HTML 파일은 표 구조가 비교적 잘 유지되므로 `pandas.read_html()`을 이용하여 환율 테이블을 읽는다.  
이때 `lxml` 파서를 명시적으로 사용하여 파서 관련 오류를 줄인다.

각 HTML 파일에서 날짜와 매매기준율 컬럼만 추출하고, 날짜를 인덱스로 설정한 뒤 컬럼명을 통화명으로 지정한다

In [ ]:
def parse_html_currency(file_path: Path, currency: str) -> pd.DataFrame:
    tables = pd.read_html(file_path, flavor="lxml")

    target = None
    for tbl in tables:
        cols = [str(c).strip() for c in tbl.columns]
        joined = " ".join(cols)

        if ("일자" in joined or "조회일" in joined) and ("매매기준율" in joined):
            target = tbl.copy()
            break

    if target is None:
        raise ValueError(f"{file_path}에서 환율 테이블을 찾지 못했습니다.")

    target.columns = [str(c).strip() for c in target.columns]

    date_col = None
    rate_col = None

    for c in target.columns:
        if "일자" in c or "조회일" in c:
            date_col = c
        if "매매기준율" in c:
            rate_col = c

    if date_col is None or rate_col is None:
        raise ValueError(f"{file_path}에서 날짜/매매기준율 컬럼을 찾지 못했습니다.")

    df = target[[date_col, rate_col]].copy()
    df.columns = ["Date", currency]

    df["Date"] = (
        df["Date"]
        .astype(str)
        .str.extract(r"(\d{4}\.\d{2}\.\d{2})")[0]
    )

    df[currency] = (
        df[currency]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.extract(r"([0-9]+(?:\.[0-9]+)?)")[0]
        .astype(float)
    )

    df = df.dropna()
    df["Date"] = pd.to_datetime(df["Date"], format="%Y.%m.%d")
    df = df.sort_values("Date").drop_duplicates(subset="Date")
    df = df.set_index("Date")

    return df

In [ ]:
for cur in CURRENCIES:
    file_path = HTML_DIR / f"{cur}.html"
    print(f"\n===== {cur}.html =====")
    temp_df = parse_html_currency(file_path, cur)
    display(temp_df.head())

각 HTML 파일이 정상적으로 읽히는지 먼저 개별적으로 확인한다.

In [ ]:
for cur in CURRENCIES:
    file_path = HTML_DIR / f"{cur}.html"
    print(f"\n===== {cur}.html =====")
    temp_df = parse_html_currency(file_path, cur)
    display(temp_df.head())

## USD.html, EUR.html, JPY.html 파일을 각각 읽어 날짜 기준으로 병합하여 하나의 HTML 기반 통합 데이터프레임을 생성한다.

In [ ]:
def build_html_dataframe():
    dfs = []
    for cur in CURRENCIES:
        file_path = HTML_DIR / f"{cur}.html"
        dfs.append(parse_html_currency(file_path, cur))

    df_html = pd.concat(dfs, axis=1).sort_index()
    return df_html

df_html = build_html_dataframe()
display(df_html.head())

## PDF 파일은 HTML보다 구조가 덜 안정적이므로, 먼저 각 페이지의 텍스트를 줄 단위로 추출한 뒤 정규표현식을 이용하여 날짜와 환율 값을 찾는다.

## 우리은행 환율표는 한 줄에 여러 숫자가 함께 존재하므로, 그중 매매기준율에 해당하는 값을 선택하여 사용한다.

In [ ]:
def extract_lines_from_pdf(pdf_path: Path):
    lines = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                lines.extend(text.splitlines())
    return lines


def parse_pdf_currency(pdf_path: Path, currency: str) -> pd.DataFrame:
    lines = extract_lines_from_pdf(pdf_path)

    rows = []
    date_pattern = re.compile(r"^\d{4}\.\d{2}\.\d{2}")

    for line in lines:
        line = line.strip()

        # 날짜로 시작하는 줄만 사용
        if not date_pattern.match(line):
            continue

        parts = line.split()

        # 최소한 날짜 + 숫자 여러 개는 있어야 함
        if len(parts) < 7:
            continue

        date_str = parts[0]

        # 날짜 뒤 숫자들만 추출
        numeric_parts = []
        for x in parts[1:]:
            x_clean = x.replace(",", "")
            try:
                numeric_parts.append(float(x_clean))
            except ValueError:
                continue

        # 보통 마지막 숫자는 대미환산율(1.0000),
        # 그 앞 숫자가 매매기준율
        if len(numeric_parts) < 2:
            continue

        rate = numeric_parts[-2]

        rows.append((date_str, rate))

    if not rows:
        raise ValueError(f"{pdf_path}에서 날짜/환율 데이터를 추출하지 못했습니다.")

    df = pd.DataFrame(rows, columns=["Date", currency])
    df["Date"] = pd.to_datetime(df["Date"], format="%Y.%m.%d")
    df = df.sort_values("Date").drop_duplicates(subset="Date")
    df = df.set_index("Date")

    return df

PDF 파일은 인쇄 형식에 따라 구조가 다를 수 있으므로, 먼저 일부 텍스트를 확인하여 날짜와 숫자가 정상적으로 추출되는지 점검한다.

In [ ]:
sample_lines = extract_lines_from_pdf(PDF_DIR / "USD.pdf")

for i, line in enumerate(sample_lines[:30]):
    print(i, line)

In [ ]:
for cur in CURRENCIES:
    file_path = PDF_DIR / f"{cur}.pdf"
    print(f"\n===== {cur}.pdf =====")
    temp_df = parse_pdf_currency(file_path, cur)
    display(temp_df.head())

USD.pdf, EUR.pdf, JPY.pdf 파일을 각각 읽어 날짜 기준으로 병합하여 하나의 PDF 기반 통합 데이터프레임을 생성한다.

In [ ]:
def build_pdf_dataframe():
    dfs = []
    for cur in CURRENCIES:
        file_path = PDF_DIR / f"{cur}.pdf"
        dfs.append(parse_pdf_currency(file_path, cur))

    df_pdf = pd.concat(dfs, axis=1).sort_index()
    return df_pdf

df_pdf = build_pdf_dataframe()
display(df_pdf.head())

생성된 HTML 기반 데이터프레임과 PDF 기반 데이터프레임의 구조를 확인한다.  
행 인덱스는 날짜이고, 컬럼명은 USD, EUR, JPY로 구성되어야 한다.

In [ ]:
print("HTML DataFrame")
display(df_html.head())
print(df_html.info())
display(df_html.describe())

print("\nPDF DataFrame")
display(df_pdf.head())
print(df_pdf.info())
display(df_pdf.describe())

기초분석 1~7을 수행하는 함수를 정의한다.  
같은 함수를 HTML 기반 데이터프레임과 PDF 기반 데이터프레임에 각각 적용한다.

In [ ]:
def run_basic_analysis(df: pd.DataFrame, title_prefix="DATA"):
    print(f"========== {title_prefix} ORIGINAL DATA ==========")
    display(df.head())

    # 기초분석1: wide -> long -> wide
    df_reset = df.reset_index()

    long_df = pd.melt(
        df_reset,
        id_vars="Date",
        value_vars=["USD", "EUR", "JPY"],
        var_name="Currency",
        value_name="Rate"
    )

    wide_df_restored = long_df.pivot(
        index="Date",
        columns="Currency",
        values="Rate"
    ).sort_index()

    print(f"[{title_prefix}] Basic Analysis 1: Long Format")
    display(long_df.head())

    print(f"[{title_prefix}] Basic Analysis 1: Restored Wide Format")
    display(wide_df_restored.head())

    # 기초분석2: 월별 최대, 최소, 평균
    monthly_stats = df.groupby(pd.Grouper(freq="ME")).agg(["max", "min", "mean"])
    print(f"[{title_prefix}] Basic Analysis 2: Monthly Statistics")
    display(monthly_stats)

    # 기초분석3: USD_BIN 생성
    q1 = df["USD"].quantile(1/3)
    q2 = df["USD"].quantile(2/3)

    df_bin = df.copy()
    df_bin["USD_BIN"] = pd.cut(
        df_bin["USD"],
        bins=[-float("inf"), q1, q2, float("inf")],
        labels=["low", "med", "high"],
        include_lowest=True
    )

    print(f"[{title_prefix}] Basic Analysis 3: USD_BIN")
    display(df_bin.head())
    display(df_bin["USD_BIN"].value_counts())

    # 기초분석4: USD_BIN 기준 집계
    bin_agg = df_bin.groupby("USD_BIN", observed=False)[["USD", "EUR", "JPY"]].agg(["max", "min", "mean"])
    print(f"[{title_prefix}] Basic Analysis 4: Aggregation by USD_BIN")
    display(bin_agg)

    # 기초분석5: crosstab
    df_bin["Month"] = df_bin.index.to_period("M").astype(str)
    ct = pd.crosstab(df_bin["Month"], df_bin["USD_BIN"])

    print(f"[{title_prefix}] Basic Analysis 5: Crosstab")
    display(ct)

    # 기초분석6: 일별 변화 그래프
    plt.figure(figsize=(12, 6))
    plt.plot(df.index, df["USD"], label="USD")
    plt.plot(df.index, df["EUR"], label="EUR")
    plt.plot(df.index, df["JPY"], label="JPY")
    plt.title(f"{title_prefix} - Daily Exchange Rate Changes")
    plt.xlabel("Date")
    plt.ylabel("Rate")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # 기초분석7: MA, EWM
    df_ma = df.copy()
    df_ma["USD_MA_20"] = df_ma["USD"].rolling(window=20, min_periods=1).mean()
    df_ma["USD_EWM_20"] = df_ma["USD"].ewm(span=20, adjust=False).mean()

    plt.figure(figsize=(12, 6))
    plt.plot(df_ma.index, df_ma["USD"], label="USD")
    plt.plot(df_ma.index, df_ma["USD_MA_20"], label="USD_MA_20")
    plt.plot(df_ma.index, df_ma["USD_EWM_20"], label="USD_EWM_20")
    plt.title(f"{title_prefix} - USD, MA, and EWM")
    plt.xlabel("Date")
    plt.ylabel("Rate")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    return {
        "long_df": long_df,
        "wide_df_restored": wide_df_restored,
        "monthly_stats": monthly_stats,
        "df_bin": df_bin,
        "bin_agg": bin_agg,
        "crosstab": ct,
        "df_ma": df_ma
    }

In [ ]:
html_results = run_basic_analysis(df_html, title_prefix="HTML")

In [ ]:
pdf_results = run_basic_analysis(df_pdf, title_prefix="PDF")

HTML 기반 데이터프레임과 PDF 기반 데이터프레임은 동일한 원본 정보를 기반으로 생성되었지만, 데이터 추출 방식에는 차이가 존재한다.  
HTML은 표 구조를 직접 읽을 수 있어 상대적으로 안정적이며, PDF는 인쇄 형식 특성상 추가 전처리가 필요하다.

따라서 두 데이터프레임을 비교하여 데이터 수집 형식에 따른 차이를 확인한다.

In [ ]:
compare = df_html.join(df_pdf, lsuffix="_HTML", rsuffix="_PDF", how="inner")
display(compare.head())

for cur in CURRENCIES:
    diff = (compare[f"{cur}_HTML"] - compare[f"{cur}_PDF"]).abs()
    print(f"{cur} mean absolute difference: {diff.mean():.6f}")

우리은행 환율조회서비스를 이용하여 최근 1년간 USD, EUR, JPY의 일별 환율 데이터를 HTML 및 PDF 형식으로 각각 수집하였다.

이후 각 파일로부터 데이터를 추출하여 HTML 기반 통합 데이터프레임과 PDF 기반 통합 데이터프레임을 생성하였고, 이를 바탕으로 넓은 형식과 긴 형식 데이터 변환, 월별 통계 집계, USD 환율 구간 분류, 교차표 작성, 일별 시각화, 이동평균 및 지수가중 이동평균 분석을 수행하였다.

분석 결과 HTML 기반 데이터는 구조화된 표를 직접 읽을 수 있어 보다 안정적으로 처리할 수 있었으며, PDF 기반 데이터는 인쇄 형식의 특성으로 인해 추가적인 전처리가 필요함을 확인할 수 있었다.